In [10]:
from IPython.core.interactiveshell import dis
import numpy as np
import pandas as pd
url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"

df = pd.read_csv(url)
print("shape:", df.shape)
print("columns:", df.columns.tolist())
# USA = df[df["continent"] == 'North America']
# USA.head(20000)
#df.head(1000)

#filter only ghana
print("✅ Ghana data filtered!")
Ghana = df[df['location'] == 'Ghana'].copy()
print("Ghana Shape:", Ghana.shape)
#print(Ghana.info())
#print(Ghana.describe())
#print(Ghana.isnull().sum())
#pd.set_option('display.max_column', None)
#pd.set_option('display.max_rows', None)

#print("Ghana Columns:", Ghana.columns.tolist())
#display(Ghana.head(50))

# Ghana COVID Key Statistics
print("=============================")
print("🇬🇭 GHANA COVID-19 SUMMARY")
print("=============================")

total_cases = Ghana["total_cases"].max()
total_deaths = Ghana["total_deaths"].max()
total_vaccinated = Ghana["people_vaccinated"].max()
population = Ghana["population"].max()
death_rate = (total_deaths / total_cases) * 100
cases_per_million =  Ghana["total_cases_per_million"].max()

print(f"Population:        {population:,.0f}")
print(f"Total Cases:       {total_cases:,.0f}")
print(f"Total Deaths:      {total_deaths:,.0f}")
print(f"Total Vaccinated:  {total_vaccinated:,.0f}")
print(f"Death Rate:        {death_rate:.2f}%")
print(f"Cases per Million: {cases_per_million:,.0f}")
print("=============================")

#compare lastest data per country
print("==== AFRICA COMPARISON ====")
african_countries = [
    "Ghana", "Nigeria", "Kenya",
    "South Africa", "Egypt",
    "Ethiopia", "Senegal",
    "Morocco", "Togo", "Tanzania",
    "Uganda", "Rwanda", "Zimbabwe",
    "Zambia", "Mozambique", "Mali",
    "Burkina Faso", "Niger", "Chad",
    "Sudan", "Libya", "Tunisia",
    "Algeria", "Cameroon", "Angola",
    "Ivory Coast", "Madagascar",
    "Malawi", "Guinea", "Benin",
    "Sierra Leone", "Botswana",
    "Namibia", "Gabon", "Gambia",
    "Mauritius", "Eswatini", "Lesotho",
    "Liberia", "Guinea-Bissau",
    "Equatorial Guinea", "Djibouti",
    "Comoros", "Cape Verde",
    "Sao Tome and Principe",
    "Seychelles", "Somalia",
    "South Sudan", "Eritrea",
    "Burundi", "Central African Republic",
    "Democratic Republic of Congo",
    "Republic of Congo"
]

africa = df[df["location"].isin(
    african_countries)]
#Get the lastest data per year
lastest = africa.groupby("location").last().reset_index()

comparison = lastest[[
    "location",
    "population",
    "total_cases",
    "total_deaths",
    "people_vaccinated",
    "total_cases_per_million"
]].sort_values("total_cases", ascending = False)

print(comparison.to_string(index = False))
print("=============================")

#Death Rate Comparison
print("=== DEATH RATE COMPARISON ====")
lastest["death_rate"] = (lastest["total_deaths"]
                         / lastest["total_cases"] * 100).round(2)
death_comparison = lastest[[
    "location",
    "death_rate"
]].sort_values("death_rate", ascending =False)

print(death_comparison.to_string(index=False))
print("\nGhana death rate:",
lastest[lastest["location"] == "Ghana"]
 ["death_rate"].values[0], "%" )
print("============================")

print("Best Country:",
      death_comparison.iloc[-1]["location"],
      "→",
      death_comparison.iloc[-1]["death_rate"],
      "%")
print("Worst Country:",
      death_comparison.iloc[0]["location"],
      "→",
      death_comparison.iloc[0]["death_rate"],
      "%")
#CASES OVER TIME IN GHANA
print("==== CASES OVER TIME IN GHANA ====")
#CLEAN DATA
Ghana_time = Ghana[["date", "new_cases", "new_deaths"]].dropna()

#CONVERT DATE
Ghana_time["date"] = pd.to_datetime(Ghana_time["date"])

#FIND THE PEAK DAY
peak_cases_idx= Ghana_time["new_cases"].idxmax()
peak_day = Ghana_time.loc[peak_cases_idx]

print(f"Peak cases day:, {peak_day["date"].strftime('%B %d, %Y')}")
print(f"Cases on the peak day:, {peak_day["new_cases"]:,.0f}")

#CASES BY YEAR
Ghana_time["year"] = Ghana_time["date"].dt.year

yearly = Ghana_time.groupby("year")["new_cases"].sum()

print("\n==== CASES BY YEAR ====")
for year, cases in yearly.items():
  print(f"{year}: {cases:,.0f}")

#VACCINATION ANALYSIS
print("==== VACCINATION ANALYSIS ====")
#CLEAN VACCINATION DATA
vacc_date = Ghana[["date", "people_vaccinated", "people_fully_vaccinated"]].dropna()

vacc_date["date"] = pd.to_datetime(vacc_date["date"])

#When did vaccinatin start
first_vacc = vacc_date[vacc_date["people_vaccinated"] > 0].iloc[0]
print(f"First vaccination day: {first_vacc["date"].strftime('%B %d, %Y')}")

#final vaccination number
total_vacc = vacc_date["people_vaccinated"].max()
fully_vacc = vacc_date["people_fully_vaccinated"].max()
population = Ghana["population"].max()
vacc_rate = (total_vacc / population) * 100
full_vacc_rate = (fully_vacc / population) * 100

print(f"Total vaccinated: {total_vacc:,.0f}")
print(f"Fully vaccinated: {fully_vacc:,.0f}")
print(f"Vaccination rate: {vacc_rate:.1f}%")
print(f"Full vacc rate: {full_vacc_rate:.1f}%")

#Did vaccination reduce deaths?
Ghana_vacc = Ghana.copy()
Ghana_vacc["date"] = pd.to_datetime(Ghana_vacc["date"])
Ghana_vacc["year"] = Ghana_vacc["date"].dt.year

yearly_deaths = Ghana_vacc.groupby("year")["new_deaths"].sum()


print("\n=== DEATHS BY YEAR ===")
for year, deaths in yearly_deaths.items():
    print(f"{year}: {deaths:,.0f} deaths")

print("=" * 40)
print("🇬🇭 GHANA COVID-19 FINAL SUMMARY")
print("=" * 40)

print(f"""
OVERVIEW:
→ Population:        33,475,870
→ Total Cases:          172,062
→ Total Deaths:           1,462
→ Death Rate:             0.85%

PEAK:
→ Worst day:   Dec 26, 2021
→ Worst year:  2021 (86,614 cases)

VACCINATION:
→ Started:     May 31, 2021
→ Vaccinated:  13,864,186 (41.4%)
→ Fully vacc:  10,780,003 (32.2%)

VACCINE IMPACT:
→ Deaths before vaccines: 1,287
→ Deaths after vaccines:    175
→ Lives saved:            86.4%

AFRICA COMPARISON:
→ Ghana ranked 17th in cases
→ Ghana ranked #11 best death rate among ALL African countries!
→  Burundi 1st the best deathe rate among All Afriacn countries

KEY FINDING:
→ Ghana handled COVID well
→ Vaccines dramatically
   reduced deaths
→ By 2024 COVID eliminated
   in Ghana ✅
""")
print("=" * 40)
print("Analysis by Solomon Nanleeb")
print("Aspiring AI Scientist 🚀")
print("UMAT Ghana 🌍")
print("=" * 40)

















shape: (429435, 67)
columns: ['iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases', 'new_cases_smoothed', 'total_deaths', 'new_deaths', 'new_deaths_smoothed', 'total_cases_per_million', 'new_cases_per_million', 'new_cases_smoothed_per_million', 'total_deaths_per_million', 'new_deaths_per_million', 'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients', 'icu_patients_per_million', 'hosp_patients', 'hosp_patients_per_million', 'weekly_icu_admissions', 'weekly_icu_admissions_per_million', 'weekly_hosp_admissions', 'weekly_hosp_admissions_per_million', 'total_tests', 'new_tests', 'total_tests_per_thousand', 'new_tests_per_thousand', 'new_tests_smoothed', 'new_tests_smoothed_per_thousand', 'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'total_boosters', 'new_vaccinations', 'new_vaccinations_smoothed', 'total_vaccinations_per_hundred', 'people_vaccinated_per_hundred', 'people_fully_vac